# Module 07 — Perplexity

Module 06 scored its bigram model with the average negative log-likelihood
(NLL) over the training bigrams. That number works, but it's not very
intuitive on its own — "2.52 nats" doesn't mean much at a glance.
**Perplexity** is just `exp(NLL)`, and it has a genuinely useful
interpretation: *a perplexity of P means the model is, on average, as
uncertain about what comes next as if it had to guess uniformly among P
equally likely options.* Lower perplexity = more confident, correct
predictions.

This module also fixes a gap Module 06 left open: it only ever evaluated
the model on the exact same data it was counted from. Here we split into
train/test and check whether the model actually **generalizes** to names
it never saw — the real question any language model evaluation cares
about.

## 1. Rebuild Module 06's bigram model (same as Module 06, not
re-explained here) — but now on a train/test split

In [ ]:
import random

import torch

names = [
    "aether", "lumine", "amber", "kaeya", "lisa", "jean", "barbara", "diluc",
    "noelle", "bennett", "fischl", "sucrose", "chongyun", "klee", "xingqiu",
    "ningguang", "beidou", "xiangling", "xiao", "zhongli", "hutao", "yanfei",
    "rosaria", "albedo", "diona", "mona", "keqing", "qiqi", "venti",
    "tartaglia", "ganyu", "xinyan", "sayu", "kokomi", "kazuha", "ayaka",
    "yoimiya", "sara", "raiden", "aloy", "itto", "gorou", "yaemiko",
    "shinobu", "heizou", "yelan", "tighnari", "nahida", "nilou", "cyno",
    "candace", "layla", "wanderer", "faruzan", "dehya", "mika", "kaveh",
    "baizhu", "kirara", "lynette", "lyney", "freminet", "neuvillette",
    "wriothesley", "charlotte", "furina", "chevreuse", "navia", "chiori",
    "arlecchino", "clorinde", "sigewinne", "emilie", "kachina", "kinich",
    "mualani", "xilonen", "ororon", "chasca", "mavuika", "citlali", "varesa",
    "iansan", "escoffier", "ineffa",
]

random.seed(42)
shuffled = names[:]
random.shuffle(shuffled)
split = int(0.8 * len(shuffled))
train_names, test_names = shuffled[:split], shuffled[split:]
print(f"{len(train_names)} train names, {len(test_names)} test names")

chars = sorted(set("".join(names)))
vocab = ["."] + chars
stoi = {ch: i for i, ch in enumerate(vocab)}


def count_bigrams(name_list):
    N = torch.zeros((len(vocab), len(vocab)), dtype=torch.int32)
    for name in name_list:
        wrapped = "." + name + "."
        for ch1, ch2 in zip(wrapped, wrapped[1:]):
            N[stoi[ch1], stoi[ch2]] += 1
    return N


N_train = count_bigrams(train_names)
P_train = (N_train + 1).float()
P_train = P_train / P_train.sum(dim=1, keepdim=True)

## 2. From NLL to perplexity

In [ ]:
def nll(name_list, P):
    log_likelihood = 0.0
    n = 0
    for name in name_list:
        wrapped = "." + name + "."
        for ch1, ch2 in zip(wrapped, wrapped[1:]):
            log_likelihood += torch.log(P[stoi[ch1], stoi[ch2]]).item()
            n += 1
    return -log_likelihood / n


def perplexity(name_list, P):
    return torch.exp(torch.tensor(nll(name_list, P))).item()


train_ppl = perplexity(train_names, P_train)
print(f"Train perplexity: {train_ppl:.2f}")
print(f"(For reference, a uniform-random model over {len(vocab)} characters has perplexity {len(vocab)}.)")

## 3. The real test: perplexity on unseen names

This is the actual point of the exercise. The bigram counts were built
*only* from `train_names`. Does the resulting probability table still make
reasonable predictions on `test_names` — names it never counted at all?

In [ ]:
test_ppl = perplexity(test_names, P_train)
print(f"Train perplexity: {train_ppl:.2f}")
print(f"Test perplexity:  {test_ppl:.2f}")

assert test_ppl > train_ppl, "test perplexity should be higher (worse) than train - the model fits training data best"
print("\nAs expected, test perplexity is higher: the model fits what it counted better than genuinely new data.")

## 4. Why smoothing matters: unsmoothed perplexity can be infinite

Add-one smoothing in Module 06 wasn't just a nicety — without it, any
bigram that never appeared in training gets probability exactly 0, and
`log(0) = -inf`. A single unseen bigram in the test set would make the
*entire* perplexity infinite. Let's see it happen.

In [ ]:
P_unsmoothed = N_train.float()
row_sums = P_unsmoothed.sum(dim=1, keepdim=True)
row_sums[row_sums == 0] = 1  # avoid 0/0 for characters that never start a bigram in train
P_unsmoothed = P_unsmoothed / row_sums

unsmoothed_test_ppl = perplexity(test_names, P_unsmoothed)
print(f"Unsmoothed test perplexity: {unsmoothed_test_ppl}")
assert unsmoothed_test_ppl == float("inf"), "expected at least one never-seen bigram in the test set to zero out the probability"
print("\nConfirmed: at least one bigram in the test names never appeared in training, so the unsmoothed model assigns it probability 0 and perplexity blows up to infinity. This is exactly why Module 06 added +1 to every count.")

## Recap

- **Perplexity = exp(average NLL)** — the same information as NLL, just on
  a scale that means something ("as confused as guessing among P
  options").
- The number that actually matters is perplexity on **held-out** data, not
  training data — otherwise a model that just memorizes gets a perfect
  score while learning nothing generalizable.
- Smoothing isn't optional bookkeeping: without it, a single unseen
  bigram/word at evaluation time makes perplexity infinite. This exact
  problem is why production tokenizers and models use much larger training
  sets and smarter smoothing/subword strategies — the more you've seen,
  the less likely you hit a true zero.

Every model built for the rest of this project — the neural n-gram model,
nanoGPT, and eventually the real pretrain — gets evaluated with perplexity
on held-out data, exactly like this.